# A-Vision (AvitoTech) в Google Colab

[Модель на HF](https://huggingface.co/AvitoTech/avision) — VLM: фото + текстовый промпт.

**GPU:** *Среда выполнения → T4 GPU*.

**Почему OOM на T4:** ~7.4B в bf16 ≈ **14+ GB только весов** — на **14.5 GB** видеопамяти почти не остаётся места под активации → `CUDA out of memory`. В ноутбуке ниже по умолчанию включена загрузка в **4-bit** (`bitsandbytes`) — обычно **~5–6 GB** VRAM. Если сессия Colab уже что-то заняла GPU, выполни **Среда → Перезапустить сессию** и ячейки заново.

**Токен HF:** не вставляй в ячейки; в Colab — *Секреты* → `HF_TOKEN`, затем `userdata.get("HF_TOKEN")`. Утечка токена → отзови в [настройках HF](https://huggingface.co/settings/tokens).


In [ ]:
!nvidia-smi 2>/dev/null || echo "Включи GPU: Среда выполнения → T4 GPU"

In [ ]:
# Версии как в requirements-colab.txt (репозиторий 2_photo_scrapper). Без -U.
%pip install -q "transformers>=4.30.0" "accelerate>=1.1.0" "safetensors>=0.4.3" "huggingface-hub>=0.24.0" qwen-vl-utils "bitsandbytes>=0.43.0" "Pillow>=10.2.0,<12" "sentencepiece>=0.2.0"

## Картинка
Вариант A — загрузить файл с компьютера. Вариант B — скачать по URL (раскомментируй нужное).

In [ ]:
from pathlib import Path

# True — выбрать файл с компьютера; False — скачать по URL ниже
USE_UPLOAD = True

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    assert uploaded, "Выбери файл изображения"
    IMAGE_PATH = Path(list(uploaded.keys())[0])
else:
    import urllib.request
    IMAGE_PATH = Path("/content/sample.jpg")
    _url = "https://images.unsplash.com/photo-1521572163474-6864f9cf17ab?w=800"
    urllib.request.urlretrieve(_url, IMAGE_PATH)

print("Изображение:", IMAGE_PATH.resolve())

## Промпт и параметры генерации

In [ ]:
MODEL_ID = "AvitoTech/avision"
PROMPT = "Опиши изображение. Что за одежда и какой цвет?"
MAX_NEW_TOKENS = 256

# T4 / мало VRAM — True; A100 40GB можно поставить False (bf16 без квантизации)
USE_4BIT = True

# Меньше пикселей → меньше памяти на vision-часть (при всё ещё OOM снизь до 512*28*28)
MIN_PIXELS = 4 * 28 * 28
MAX_PIXELS = 768 * 28 * 28

## Загрузка модели (один раз за сессию)

После выполнения ячейки ниже в памяти остаются `model`, `processor` и функция **`avision_ask`**. Следующие вопросы — только ячейка «Ещё запрос».

In [ ]:
import gc
import os
import re
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

assert torch.cuda.is_available(), "Нужна GPU в Colab"

torch.cuda.empty_cache()
gc.collect()

# Опционально: секрет Colab с именем HF_TOKEN (для gated / лимитов)
try:
    from google.colab import userdata
    _t = userdata.get("HF_TOKEN")
    if _t:
        os.environ["HF_TOKEN"] = _t
except Exception:
    pass

_hf_kw = {}
if os.environ.get("HF_TOKEN"):
    _hf_kw["token"] = os.environ["HF_TOKEN"]

print(f"Загрузка {MODEL_ID!r} (4-bit={USE_4BIT})…")
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        **_hf_kw,
    )
else:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype="auto",
        device_map="auto",
        **_hf_kw,
    )
processor = AutoProcessor.from_pretrained(MODEL_ID, **_hf_kw)


def clean_avision_output(text: str) -> str:
    """SentencePiece/чат: убрать ▁, заменить <0x0A> на перенос, сжать пробелы."""
    if not text:
        return text
    t = text.replace("<0x0A>", "\n").replace("<0x0a>", "\n")
    t = t.replace("▁", " ")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


def avision_ask(prompt, image, max_new_tokens=None):
    """Один запрос к уже загруженной модели. image — PIL.Image или путь (str/Path)."""
    if max_new_tokens is None:
        max_new_tokens = MAX_NEW_TOKENS
    if isinstance(image, (str, Path)):
        pil = Image.open(image).convert("RGB")
    else:
        pil = image.convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": pil,
                    "min_pixels": MIN_PIXELS,
                    "max_pixels": MAX_PIXELS,
                },
                {"type": "text", "text": prompt},
            ],
        }
    ]
    chat_text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[chat_text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    in_len = inputs["input_ids"].shape[1]
    new_tokens = generated_ids[0, in_len:]
    tok = getattr(processor, "tokenizer", None) or processor
    raw = tok.decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )
    return clean_avision_output(raw)


# Первый ответ (как раньше)
print("\n--- Ответ 1 ---\n")
print(avision_ask(PROMPT, IMAGE_PATH))

### Ещё запрос (без перезагрузки модели)

Нужны только `model`, `processor` и функция **`avision_ask`** из ячейки выше (она уже должна быть выполнена). Меняй промпт и/или картинку и запускай эту ячейку.

In [ ]:
# Второй (и любой следующий) запрос — только это
PROMPT_2 = "Какие элементы одежды видишь? Список."
# та же картинка:
print(avision_ask(PROMPT_2, IMAGE_PATH))
# или другой файл: print(avision_ask(PROMPT_2, "/content/drugoe.jpg"))
# или PIL: print(avision_ask(PROMPT_2, Image.open(IMAGE_PATH)))